In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import seaborn as sns
sns.set_theme("notebook",style="dark")

In [ ]:
import torch
from qgsw.logging.core import getLogger, setup_root_logger
from qgsw.specs import defaults

from __future__ import annotations

import os
from pathlib import Path
import gc

import numpy as np
import pandas as pd
import torch
import xarray as xr
from scipy.ndimage import gaussian_filter

from qgsw.configs.core import Configuration
from qgsw.decomposition.core import build_basis_from_params_dict
from qgsw.eNATL60.fields_computations import compute_stream_function_ssh_only
from qgsw.eNATL60.interpolation import (
    build_regridder,
    compute_lonlat_from_regular_xy_grid,
    lonlat_to_xy,
)
from qgsw.eNATL60.loading import load_datasets
from qgsw.eNATL60.var_keys import (
    LATITUDE,
    LONGITUDE,
    SSH,
    STREAMFUNCTION,
    TIME,
)
from qgsw.logging.utils import box, step
from qgsw.masks import Masks
from qgsw.models.qg.psiq.core import QGPSIQ
from qgsw.physics.constants import EARTH_ANGULAR_ROTATION, EARTH_RADIUS
from qgsw.physics.coriolis.beta_plane import BetaPlane
from qgsw.solver.boundary_conditions.base import Boundaries
from qgsw.spatial.core.discretization import (
    SpaceDiscretization2D,
)
from qgsw.spatial.core.grid import Grid2D
from qgsw.utils.storage import get_path_from_env
from qgsw.utils.tensor_dict import change_specs


torch.backends.cudnn.deterministic = True

ROOT_PATH = Path(os.getcwd()).parent


specs=defaults.get()

logger=getLogger(__name__)
setup_root_logger(1)

In [ ]:
dt = 7200
n_file_per_cycle = 20
n_steps_per_cyle = 240
comparison_interval = 1
n_cycles=3

### Load filenames

In [ ]:
def sort_files_by_dates(file_paths: list[Path]) -> list[Path]:
    """Sort file names by dates."""
    times = pd.to_datetime([f.name[-20:-12] for f in file_paths])
    args = np.argsort(times)
    return np.array(file_paths)[args].tolist()


data_folder = get_path_from_env(key="DATA_FOLDER")
files = list(
    (data_folder/"MEANDERS").glob("eNATL60-BLB002_1h_2009*_2009*_gridT-2D_2009*-2009*.nc"),
)
files = sort_files_by_dates(files)

### Dataset formatting func

In [ ]:
def format_ds(ds: xr.Dataset) -> xr.Dataset:
    """Format Dataset."""
    # Drop useless variables
    if "axis_nbounds" in ds.dims:
        ds = ds.drop_dims("axis_nbounds")
    if "time_centered" in ds.coords:
        ds = ds.reset_coords("time_centered", drop=True)
    # Rename
    ds = ds.rename(
        {
            "time_counter": TIME,
            "nav_lon": LONGITUDE,
            "nav_lat": LATITUDE,
            "x": "i",
            "y": "j",
            "sossheig": SSH,
        }
    )
    ds = ds.transpose(TIME, "i", "j")
    return ds.set_coords([LONGITUDE, LATITUDE])

### Build space

In [ ]:
ds_init = load_datasets(files[0], format_func=format_ds)

dx = dy = 10000
lons, lats = compute_lonlat_from_regular_xy_grid(
    ds_init[LONGITUDE],
    ds_init[LATITUDE],
    dx=dx,
    dy=dy,
)
xs, ys = lonlat_to_xy(lons, lats)

### Compute β-plane parameters

lat0 = (lats.max() + lats.min()) / 2
beta_plane = BetaPlane(
    f0=2 * EARTH_ANGULAR_ROTATION * np.sin(lat0),
    beta=2 * EARTH_ANGULAR_ROTATION * np.cos(lat0) / EARTH_RADIUS,
)

### Build regridder

psi_regridder = build_regridder(ds_init, lons, lats)


## Areas
nx, ny = lats.shape
xx = torch.tensor(xs.round(), **specs)
space_2d = SpaceDiscretization2D.from_psi_grid(
    Grid2D(
        x=xx - xx[0, :],
        y=torch.tensor(ys.round(), **specs),
    )
)
### Boundaries offset

b = 4

space_interior = space_2d.slice(
    b,
    space_2d.psi.xy.x.shape[0] - b,
    b,
    space_2d.psi.xy.x.shape[1] - b,
)

nx = space_interior.nx
ny = space_interior.ny
dx = space_interior.dx
dy = space_interior.dy

### Parameters

In [ ]:
ROOT_PATH = Path(os.getcwd()).parent

# Simulation parameters

config = Configuration.from_toml(ROOT_PATH.joinpath("config/variational_analysis.toml"))

H = config.model.h
H1,H2,H3 = H
g_prime = config.model.g_prime
g1,g2,g3= g_prime
beta_plane = config.physics.beta_plane
bottom_drag_coef = config.physics.bottom_drag_coefficient
slip_coef = config.physics.slip_coef

y_w = space_2d.q.xy.y[0, :].unsqueeze(0)
y0 = 0.5 * space_interior.ly
beta_effect = beta_plane.beta * (y_w - y0)

In [ ]:
def load_cycle_data(c:int) -> tuple[list[torch.Tensor],list[torch.Tensor], list[torch.Tensor]]:
    if (c + 1) * n_file_per_cycle > len(files):
        msg = f"Not enough files to perform cycle {c} and above."
        logger.warning(msg)
        raise ValueError

    files_for_cycle = files[c * n_file_per_cycle : (c + 1) * n_file_per_cycle]

    ds = load_datasets(*files_for_cycle, format_func=format_ds)

    msg = f"Cycle {step(c + 1, n_cycles)}: eNATL60 data loaded."
    logger.info(box(msg, style="round"))

    ds[STREAMFUNCTION] = compute_stream_function_ssh_only(
        ds,
        g_prime[0].item(),
        remove_avg=True,
    )

    with logger.section("Filtering stream function..."):
        ds["psi_filt"] = xr.apply_ufunc(
            gaussian_filter,
            ds[STREAMFUNCTION].load(),
            kwargs={"sigma": 14},
            input_core_dims=[["i", "j"]],
            output_core_dims=[["i", "j"]],
            vectorize=True,
        )

    with logger.section("Interpolating stream function..."):
        regridded_psi: xr.DataArray = psi_regridder(ds[STREAMFUNCTION])
        regridded_psi_filt: xr.DataArray = psi_regridder(ds["psi_filt"])
        ds_interp = xr.Dataset(
            {
                LONGITUDE: (["i", "j"], lons),
                LATITUDE: (["i", "j"], lats),
                STREAMFUNCTION: ([TIME, "i", "j"], regridded_psi.data),
                "psi_filt": ([TIME, "i", "j"], regridded_psi_filt.data),
            },
            regridded_psi_filt.coords,
        )
        ds_interp = ds_interp.set_coords([LONGITUDE, LATITUDE])
        ds_interp = ds_interp.load()

    psis_ref = [
        torch.tensor(p, **specs).unsqueeze(0).unsqueeze(0)/beta_plane.f0
        for p in ds_interp[STREAMFUNCTION].to_numpy()
    ]
    psis_filt = [
        torch.tensor(p, **specs).unsqueeze(0).unsqueeze(0)/beta_plane.f0
        for p in ds_interp["psi_filt"].to_numpy()
    ]
    t0 = ds_interp[TIME][0]
    times = (ds_interp[TIME] - t0).dt.total_seconds().to_numpy()
    times = torch.tensor(times, **specs)

    return psis_ref,psis_filt,times

In [ ]:
def xy_to_lonlat(x:np.ndarray, y:np.ndarray, lat0:float) -> tuple[np.ndarray, np.ndarray]:
    lats = (y/EARTH_RADIUS)+lat0
    lons = x/EARTH_RADIUS/np.cos(lats)
    return lons,lats

In [ ]:
def format_dsu(ds: xr.Dataset) -> xr.Dataset:
    """Format Dataset."""
    # Drop useless variables
    if "axis_nbounds" in ds.dims:
        ds = ds.drop_dims("axis_nbounds")
    if "time_centered" in ds.coords:
        ds = ds.reset_coords("time_centered", drop=True)
    # Rename
    ds = ds.rename(
        {
            "time_counter": TIME,
            "nav_lon": LONGITUDE,
            "nav_lat": LATITUDE,
            "x": "i",
            "y": "j",
            "sozocrtx": "u",
        }
    )
    ds = ds.transpose(TIME, "i", "j")
    return ds.set_coords([LONGITUDE, LATITUDE])
def format_dsv(ds: xr.Dataset) -> xr.Dataset:
    """Format Dataset."""
    # Drop useless variables
    if "axis_nbounds" in ds.dims:
        ds = ds.drop_dims("axis_nbounds")
    if "time_centered" in ds.coords:
        ds = ds.reset_coords("time_centered", drop=True)
    # Rename
    ds = ds.rename(
        {
            "time_counter": TIME,
            "nav_lon": LONGITUDE,
            "nav_lat": LATITUDE,
            "x": "i",
            "y": "j",
            "somecrty": "v",
        }
    )
    ds = ds.transpose(TIME, "i", "j")
    return ds.set_coords([LONGITUDE, LATITUDE])

    
space_no_xoffset=SpaceDiscretization2D.from_psi_grid(
    Grid2D(
        x=torch.tensor(xs.round(),**specs),
        y=torch.tensor(ys.round(), **specs),
    )
)

ux,uy  =space_no_xoffset.u.xy
ux=ux.cpu().numpy()
uy=uy.cpu().numpy()

lonsu, latsu = xy_to_lonlat(ux,uy,lat0)

vx,vy  =space_no_xoffset.v.xy
vx=vx.cpu().numpy()
vy=vy.cpu().numpy()

lonsv, latsv = xy_to_lonlat(vx,vy,lat0)

ufiles = list(
    (data_folder/"MEANDERS").glob("eNATL60-BLB002_1h_2009*_2009*_gridU-2D_2009*-2009*.nc"),
)
ufiles = sort_files_by_dates(ufiles)

dsu_init = load_datasets(ufiles[0], format_func=format_dsu)

u_regridder = build_regridder(dsu_init, lonsu, latsu)

vfiles = list(
    (data_folder/"MEANDERS").glob("eNATL60-BLB002_1h_2009*_2009*_gridV-2D_2009*-2009*.nc"),
)
vfiles = sort_files_by_dates(vfiles)

dsv_init = load_datasets(vfiles[0], format_func=format_dsv)

v_regridder = build_regridder(dsv_init, lonsv, latsv)

def load_cycle_uv_data(c:int) -> tuple[list[torch.Tensor],list[torch.Tensor], list[torch.Tensor]]:
    if (c + 1) * n_file_per_cycle > len(ufiles):
        msg = f"Not enough files to perform cycle {c} and above."
        logger.warning(msg)
        raise ValueError
    if (c + 1) * n_file_per_cycle > len(vfiles):
        msg = f"Not enough files to perform cycle {c} and above."
        logger.warning(msg)
        raise ValueError

    files_for_cycle = ufiles[c * n_file_per_cycle : (c + 1) * n_file_per_cycle]
    dsu = load_datasets(*files_for_cycle, format_func=format_dsu)

    with logger.section("Interpolating u..."):
        regridded_u: xr.DataArray = u_regridder(dsu["u"])
        dsu_interp = xr.Dataset(
            {
                LONGITUDE: (["i", "j"], lonsu),
                LATITUDE: (["i", "j"], latsu),
                "u": ([TIME, "i", "j"], regridded_u.data),
            },
            regridded_u.coords,
        )
        dsu_interp = dsu_interp.set_coords([LONGITUDE, LATITUDE])
        dsu_interp = dsu_interp.load()

    files_for_cycle = vfiles[c * n_file_per_cycle : (c + 1) * n_file_per_cycle]
    dsv = load_datasets(*files_for_cycle, format_func=format_dsv)

    with logger.section("Interpolating v..."):
        regridded_v: xr.DataArray = v_regridder(dsv["v"])
        dsv_interp = xr.Dataset(
            {
                LONGITUDE: (["i", "j"], lonsv),
                LATITUDE: (["i", "j"], latsv),
                "v": ([TIME, "i", "j"], regridded_v.data),
            },
            regridded_v.coords,
        )
        dsv_interp = dsv_interp.set_coords([LONGITUDE, LATITUDE])
        dsv_interp = dsv_interp.load()
    

    u_refs = [
        torch.tensor(u, **specs).unsqueeze(0).unsqueeze(0)
        for u in dsu_interp["u"].to_numpy()
    ]
    v_refs = [
        torch.tensor(v, **specs).unsqueeze(0).unsqueeze(0)
        for v in dsv_interp["v"].to_numpy()
    ]

    return u_refs, v_refs

In [ ]:
from qgsw.eNATL60.fields_computations import compute_streamfunction_with_atmospheric_pressure
from qgsw.eNATL60.forcing import load_era_interim


def load_cycle_data_atmp(c:int) -> tuple[list[torch.Tensor],list[torch.Tensor], list[torch.Tensor]]:
    if (c + 1) * n_file_per_cycle > len(files):
        msg = f"Not enough files to perform cycle {c} and above."
        logger.warning(msg)
        raise ValueError


    files_for_cycle = files[c * n_file_per_cycle : (c + 1) * n_file_per_cycle]

    ds = load_datasets(*files_for_cycle, format_func=format_ds)
    ds_era = load_era_interim(data_folder / "misc", 2009)

    msg = f"Cycle {step(c + 1, n_cycles)}: eNATL60 data loaded."
    logger.info(box(msg, style="round"))

    ds[STREAMFUNCTION] = compute_streamfunction_with_atmospheric_pressure(
        ds,
        ds_era,
        1026.0,
        g_prime[0].item(),
        remove_avgs=True,
    )

    with logger.section("Filtering stream function..."):
        ds["psi_filt"] = xr.apply_ufunc(
            gaussian_filter,
            ds[STREAMFUNCTION].load(),
            kwargs={"sigma": 14},
            input_core_dims=[["i", "j"]],
            output_core_dims=[["i", "j"]],
            vectorize=True,
        )

    with logger.section("Interpolating stream function..."):
        regridded_psi: xr.DataArray = psi_regridder(ds[STREAMFUNCTION])
        regridded_psi_filt: xr.DataArray = psi_regridder(ds["psi_filt"])
        ds_interp = xr.Dataset(
            {
                LONGITUDE: (["i", "j"], lons),
                LATITUDE: (["i", "j"], lats),
                STREAMFUNCTION: ([TIME, "i", "j"], regridded_psi.data),
                "psi_filt": ([TIME, "i", "j"], regridded_psi_filt.data),
            },
            regridded_psi_filt.coords,
        )
        ds_interp = ds_interp.set_coords([LONGITUDE, LATITUDE])
        ds_interp = ds_interp.load()

    psis_ref = [
        torch.tensor(p, **specs).unsqueeze(0).unsqueeze(0)/beta_plane.f0
        for p in ds_interp[STREAMFUNCTION].to_numpy()
    ]
    psis_filt = [
        torch.tensor(p, **specs).unsqueeze(0).unsqueeze(0)/beta_plane.f0
        for p in ds_interp["psi_filt"].to_numpy()
    ]
    t0 = ds_interp[TIME][0]
    times = (ds_interp[TIME] - t0).dt.total_seconds().to_numpy()
    times = torch.tensor(times, **specs)

    return psis_ref,psis_filt,times

### Error

In [ ]:
from qgsw.solver.finite_diff import grad_perp


def rmse(f: torch.Tensor, f_ref: torch.Tensor) -> float:
    """RMSE."""
    return ((f - f_ref).square().mean() / f_ref.square().mean()).sqrt()

def uv_rmse(psi:torch.Tensor,u_ref:torch.Tensor, v_ref:torch.Tensor) -> torch.Tensor:
    """Gradient RMSE."""
    u,v = grad_perp(psi)
    
    u/=dy
    v/=dx


    return ((u-u_ref).square().mean()+(v-v_ref).square().mean()).sqrt() / (u_ref.square().mean()+v_ref.square().mean()).sqrt()

### Boundary conditions

In [ ]:
def extract_psi_bc(psi: torch.Tensor) -> Boundaries:
    """Extract psi."""
    return Boundaries.extract(psi, b, -b - 1, b, -b - 1, 2)

## Model Wrappers

In [ ]:
from collections.abc import Callable
from pathlib import Path
from typing import TypeVar

from matplotlib import pyplot as plt
import numpy as np

from qgsw.analysis.qg_model import ModelWrapper, ModelsManager
from qgsw.models.qg.psiq.core import QGPSIQCore
from qgsw.spatial.core.discretization import SpaceDiscretization2D

T = TypeVar("T", bound=QGPSIQCore)

class ModelWrapperOBC(ModelWrapper[T]):
    results_paths = Path("../output/g5k/param_optim")

    losses:dict[str,list[list[torch.Tensor]]] = {}
    
    show=True

    def __init__(self, space_2d: SpaceDiscretization2D) -> None:
        super().__init__(space_2d)
        self.losses = {
            "rmse": [],
            "uv_rmse": [],
        }

    def _set_params(self) -> None:
        space = self.model.space
        self.model.masks = Masks.empty_tensor(
            space.nx,
            space.ny,
            device=specs["device"],
        )
        self.model.bottom_drag_coef = 0
        self.model.wide = True
        self.model.slip_coef = slip_coef
        self.model.dt = dt
        
    def load(self)-> dict:
        file = self.results_paths.joinpath(self.prefix+".pt")
        return torch.load(file)
    def new_cycle(self) -> None:
        super().new_cycle()
        for loss in self.losses.values():
            loss.append([])
            
    def add_loss(self, loss_value:float,loss_name:str) -> None:
        self.losses[loss_name][-1].append(loss_value)
        
    def plot_loss(self,*,loss_name:str,ax:plt.Axes|None=None,cycle:int|None=None) -> None:
        if not self.show:
            return
        if ax is None:
            ax = plt.gca()
        cycles = [cycle] if cycle is not None else list(range(len(self.losses[loss_name])))
        time_offset= 0
        for i,c in enumerate(cycles):
            times = self.model.dt*np.arange(len(self.losses[loss_name][c]))/3600/24 + time_offset
            time_offset = times[-1] + self.model.dt/3600/24
            loss =  np.array(self.losses[loss_name][c])
            kwargs = self.plot_kwargs.copy()
            if i!= 0:
                kwargs.pop("label")
            ax.plot(times, loss, **kwargs)
        ax.set_xlabel("Time [day]")
    def step(self) -> None:
        super().step()
            
        
M = TypeVar("M", bound=ModelWrapperOBC[QGPSIQCore])

class ModelsManagerOBC(ModelsManager[M]):

    loss_fn: dict[str, Callable[[torch.Tensor,torch.Tensor], torch.Tensor]]= {
        "rmse":rmse,
    }

    losses = list(loss_fn.keys())

    def compute_loss(self, psi_ref:torch.Tensor) -> None:
        for loss_name in self.losses:
            self.loop_over_models(
                lambda mw: mw.add_loss(self.loss_fn[loss_name](mw.model.psi[0,0],psi_ref[0,0]).cpu().item(),loss_name)
            )

    def compute_uv_losses(self,u_ref:torch.Tensor, v_ref:torch.Tensor) -> None:
        self.loop_over_models(
            lambda mw: mw.add_loss(uv_rmse(mw.model.psi[0,0],u_ref[0,0],v_ref[0,0]).cpu().item(),"uv_rmse")
        )
        
        
    
    def plot_loss(self,*,loss_name:str,ax:plt.Axes|None=None,cycle:int|None=None) -> None:
        self.loop_over_models(lambda mw: mw.plot_loss(loss_name=loss_name,ax=ax,cycle=cycle))

### Reduced Gravity

In [ ]:
from torch import Tensor
from qgsw.solver.finite_diff import laplacian
from qgsw.spatial.core.discretization import SpaceDiscretization2D
from qgsw.spatial.core.grid_conversion import interpolate
from qgsw.utils.interpolation import QuadraticInterpolation
from qgsw.utils.reshaping import crop


class ReducedGravity(ModelWrapperOBC[QGPSIQ]):
    prefix = None
    color = "blue"
    label="Reduced Gravity"
    def _init_model(self, space_2d:SpaceDiscretization2D) -> None:
        self.model= QGPSIQ(
            space_2d=space_2d,
            H=H[:1],
            beta_plane=beta_plane,
            g_prime=g_prime[:1]*g_prime[1:2]/(g_prime[:1]+g_prime[1:2]),
        )
        self._set_params()
    def compute_q(self,psi: Tensor, beta_effect:torch.Tensor) -> Tensor:
        return interpolate(laplacian(psi,dx,dy) - beta_plane.f0**2 * (1/H1/g1+1/H1/g2)*psi[...,1:-1,1:-1]) + beta_effect
    
    def setup(self, psis: list[torch.Tensor],times:list[torch.Tensor],beta_effect_w:torch.Tensor) -> None:
        psi0 = psis[0]
        psi_bcs = [extract_psi_bc(psi[:,:1]) for psi in psis]
        q_bcs = [
            Boundaries.extract(
                self.compute_q(psi[:, :1],beta_effect_w), 2, -3, 2, -3, 3
            )
            for psi in psis
        ]
        self.model.set_boundary_maps(QuadraticInterpolation(times, psi_bcs), QuadraticInterpolation(times, q_bcs))
        self.model.set_psiq(crop(psi0[:,:1],b), crop(self.compute_q(psi0[:,:1],beta_effect_w),b-1))

### ψ₂ transported

In [ ]:
from qgsw.decomposition.coefficients import DecompositionCoefs
from qgsw.models.qg.psiq.modified.forced import QGPSIQRGPsi2TransportDR
from qgsw.models.qg.stretching_matrix import compute_A_tilde


class RGPsi2Transport(ModelWrapperOBC[QGPSIQRGPsi2TransportDR]):
    prefix = "results_enatl60"
    color="navy"
    label="GaussBarotropic - RG - DR"
    save_video = False
    def _init_model(self, space_2d:SpaceDiscretization2D) -> None:
        self.model= QGPSIQRGPsi2TransportDR(
            space_2d=space_2d,
            H=H[:2],
            beta_plane=beta_plane,
            g_prime=g_prime[:2],
        )
        self._set_params()
        self.alphas = {}
        self.coefs = {}
    def compute_q(self,psi: Tensor, A11:torch.Tensor, beta_effect:torch.Tensor) -> Tensor:
        return interpolate(
            laplacian(psi, dx, dy)
            - beta_plane.f0**2 * A11 * psi[..., 1:-1, 1:-1]
        ) + beta_effect
    def setup(self, psis: list[Tensor], times: list[Tensor], beta_effect_w: Tensor) -> None:
        res = self.load()

        try:
            alpha:torch.Tensor = res[self.cycle]["alpha"]
        except KeyError:
            alpha=torch.tensor(0,**specs)
        self.A = compute_A_tilde(H[:2],g_prime[:2],alpha,**specs)
        coefs = DecompositionCoefs.from_dict(change_specs(res[self.cycle]["coefs"],**specs))

        basis = build_basis_from_params_dict(res[self.cycle]["config"]["basis"])
        try:
            basis.freeze_time_normalization(self.model.dt*torch.tensor([n_steps_per_cyle],**specs))
        except:... 
        basis.set_coefs(coefs)
        self._fpsi2 = basis.localize(
            space_interior.psi.xy.x,space_interior.psi.xy.y
        )

        if self.save_params:
            self.alphas[self.cycle] = alpha
            self.coefs[self.cycle] = coefs
        psi0 = psis[0]
        psi_bcs = [extract_psi_bc(psi[:,:1]) for psi in psis]
        q_bcs = [
                Boundaries.extract(
                    self.compute_q(psi[:, :1],self.A[:1,:1],beta_effect_w), 2, -3, 2, -3, 3
                )
                for psi in psis
            ]

        self.model.set_psiq(crop(psi0[:,:1],b), crop(self.compute_q(psi0[:,:1],self.A[:1,:1],beta_effect_w),b-1))
        self.model.alpha = alpha
        self.model.basis = basis
        self.model.set_boundary_maps(QuadraticInterpolation(times, psi_bcs), QuadraticInterpolation(times, q_bcs))

### Forced

In [ ]:
from qgsw.models.qg.psiq.modified.forced import QGPSIQForced

Heq = H[:1]*H[1:2]/(H[:1]+H[1:2])
class ForcedRGDR(ModelWrapperOBC[QGPSIQForced]):
    prefix = "results_enatl60_forced"
    color="brown"
    label="Forced DR"
    save_video = False
    def _init_model(self, space_2d:SpaceDiscretization2D) -> None:
        self.model= QGPSIQForced(
            space_2d=space_2d,
            H=Heq,
            beta_plane=beta_plane,
            g_prime=g_prime[1:2],
        )
        self.model.wind_scaling = H[:1].item()
        self._set_params()
        self.coefs = {}
    def compute_q(self,psi: Tensor, beta_effect:torch.Tensor) -> Tensor:
        return interpolate(
            laplacian(psi,dx,dy)
            - beta_plane.f0**2 * (1/Heq/g2)*psi[...,1:-1,1:-1]
        ) + beta_effect

    def setup(self, psis: list[Tensor], times: list[Tensor], beta_effect_w: Tensor) -> None:
        res = self.load()
        coefs = DecompositionCoefs.from_dict(change_specs(res[self.cycle]["coefs"],**specs))
        self.basis = build_basis_from_params_dict(res[self.cycle]["config"]["basis"])
        self.basis.set_coefs(coefs)
        if self.save_params:
            self.coefs[self.cycle] = coefs
            
        self.wv = self.basis.localize(
            self.model.space.remove_h().q.xy.x,
            self.model.space.remove_h().q.xy.y,
        )
        psi0 = psis[0]
        psi_bcs = [extract_psi_bc(psi[:,:1]) for psi in psis]
        q_bcs = [
                Boundaries.extract(
                    self.compute_q(psi[:, :1],beta_effect_w), 2, -3, 2, -3, 3
                )
                for psi in psis
            ]

        self.model.set_psiq(crop(psi0[:,:1],b), crop(self.compute_q(psi0[:,:1],beta_effect_w),b-1))
        self.model.set_boundary_maps(QuadraticInterpolation(times, psi_bcs), QuadraticInterpolation(times, q_bcs))
        
    def step(self) -> None:
        self.model.forcing = self.wv(self.model.time)
        super().step()

In [ ]:
rg = ReducedGravity(space_interior)

forced_g1000 = ForcedRGDR(space_interior)
forced_g1000.prefix = "results_enatl60_forced_gamma1000_nowind_obstrack"
forced_g1000.linestyle  ="solid"
forced_g1000.label = "Forced - 1000"

forced_g1 = ForcedRGDR(space_interior)
forced_g1.prefix = "results_enatl60_forced_nowind_obstrack"
forced_g1.linestyle  ="dashed"
forced_g1.label = "Forced - 1"

forced_noreg = ForcedRGDR(space_interior)
forced_noreg.prefix = "results_enatl60_forced_noreg_nowind_obstrack"
forced_noreg.linestyle  ="dotted"
forced_noreg.label = "Forced - NR"

p2_g0_01 = RGPsi2Transport(space_interior)
p2_g0_01.prefix = "results_enatl60_gamma0_01_nowind_obstrack"
p2_g0_01.linestyle  ="solid"
p2_g0_01.label = "ψ₂ - 0.01"

p2_g0_0001 = RGPsi2Transport(space_interior)
p2_g0_0001.prefix = "results_enatl60_gamma0_0001_nowind_obstrack"
p2_g0_0001.linestyle  ="dashed"
p2_g0_0001.label = "ψ₂ - 0.0001"

p2_noreg = RGPsi2Transport(space_interior)
p2_noreg.prefix = "results_enatl60_noreg_nowind_obstrack"
p2_noreg.linestyle  ="dotted"
p2_noreg.label = "ψ₂ - NR"

p2_noalpha_g0_01 = RGPsi2Transport(space_interior)
p2_noalpha_g0_01.prefix = "results_enatl60_noalpha_gamma0_01_nowind_obstrack"
p2_noalpha_g0_01.linestyle  ="solid"
p2_noalpha_g0_01.color="lightblue"
p2_noalpha_g0_01.label = "ψ₂ - NoAlpha - 0.01"

p2_noalpha_g0_0001 = RGPsi2Transport(space_interior)
p2_noalpha_g0_0001.prefix = "results_enatl60_noalpha_gamma0_0001_nowind_obstrack"
p2_noalpha_g0_0001.linestyle  ="dashed"
p2_noalpha_g0_0001.color="lightblue"
p2_noalpha_g0_0001.label = "ψ₂ - NoAlpha - 0.0001"

p2_noalpha_noreg = RGPsi2Transport(space_interior)
p2_noalpha_noreg.prefix = "results_enatl60_noalpha_noreg_nowind_obstrack"
p2_noalpha_noreg.linestyle  ="dotted"
p2_noalpha_noreg.color="lightblue"
p2_noalpha_noreg.label = "ψ₂ - NoAlpha - NR"


models = ModelsManagerOBC(
    rg,
    forced_g1000,
    forced_g1,
    forced_noreg,
    p2_g0_01,
    p2_g0_0001,
    p2_noreg,
    p2_noalpha_g0_01,
    p2_noalpha_g0_0001,
    p2_noalpha_noreg,
)
models.save_params = True


gc.collect()
for c in range(n_cycles):
    torch.cuda.reset_peak_memory_stats()
    models.new_cycle()

    psis_ref,psis_filt, times=load_cycle_data(c)

    u_refs, v_refs = load_cycle_uv_data(c)

    psi0 = psis_filt[0]


    models.reset_time()

    models.setup(psis_filt,times,beta_effect[:,1:-1])

    models.compute_loss(crop(psis_ref[0],b))
    models.compute_uv_losses(crop(u_refs[0],b),crop(v_refs[0],b))
    for n in range(1,n_steps_per_cyle):
        models.step()
        models.compute_loss(crop(psis_ref[n],b))
        models.compute_uv_losses(crop(u_refs[n],b),crop(v_refs[n],b))

    torch.cuda.empty_cache()
    gc.collect()

    max_mem = torch.cuda.max_memory_allocated() / 1024 / 1024
    msg_mem = f"Cycle {step(c + 1, n_cycles)} | Max memory allocated: {max_mem:.1f} MB."
    logger.info(box(msg_mem, style="round"))


In [ ]:
from matplotlib import pyplot as plt

from qgsw import plots

# To Show

# Plots

show_grad = True

fig,axs = plots.subplots(1+show_grad,1,figsize=(15,5+5*show_grad))
plots.set_rowtitles(["RMSE"]+show_grad*[ "Gradient RMSE"] ,axs=axs)
models.plot_loss(loss_name="rmse",ax=axs[0,0])
plots.clamp_ylims(0,1,axs[0,0])
axs[0,0].legend(loc="upper left",prop={'size': 8})
if show_grad:
    models.plot_loss(loss_name="uv_rmse",ax=axs[1,0])
    plots.clamp_ylims(0,1,axs[1,0])
    axs[1,0].legend(loc="upper left",prop={'size': 8})

In [ ]:
rg = ReducedGravity(space_interior)

forced_g1000 = ForcedRGDR(space_interior)
forced_g1000.prefix = "results_enatl60_forced_atmp_gamma1000_nowind_obstrack"
forced_g1000.linestyle  ="solid"
forced_g1000.label = "Forced - 1000"

forced_g1 = ForcedRGDR(space_interior)
forced_g1.prefix = "results_enatl60_forced_atmp_nowind_obstrack"
forced_g1.linestyle  ="dashed"
forced_g1.label = "Forced - 1"

forced_noreg = ForcedRGDR(space_interior)
forced_noreg.prefix = "results_enatl60_forced_atmp_noreg_nowind_obstrack"
forced_noreg.linestyle  ="dotted"
forced_noreg.label = "Forced - NR"

p2_g0_01 = RGPsi2Transport(space_interior)
p2_g0_01.prefix = "results_enatl60_atmp_gamma0_01_nowind_obstrack"
p2_g0_01.linestyle  ="solid"
p2_g0_01.label = "ψ₂ - 0.01"

p2_g0_0001 = RGPsi2Transport(space_interior)
p2_g0_0001.prefix = "results_enatl60_atmp_gamma0_0001_nowind_obstrack"
p2_g0_0001.linestyle  ="dashed"
p2_g0_0001.label = "ψ₂ - 0.0001"

p2_noreg = RGPsi2Transport(space_interior)
p2_noreg.prefix = "results_enatl60_atmp_noreg_nowind_obstrack"
p2_noreg.linestyle  ="dotted"
p2_noreg.label = "ψ₂ - NR"

p2_noalpha_g0_01 = RGPsi2Transport(space_interior)
p2_noalpha_g0_01.prefix = "results_enatl60_atmp_noalpha_gamma0_01_nowind_obstrack"
p2_noalpha_g0_01.linestyle  ="solid"
p2_noalpha_g0_01.color="lightblue"
p2_noalpha_g0_01.label = "ψ₂ - NoAlpha - 0.01"

p2_noalpha_g0_0001 = RGPsi2Transport(space_interior)
p2_noalpha_g0_0001.prefix = "results_enatl60_atmp_noalpha_gamma0_0001_nowind_obstrack"
p2_noalpha_g0_0001.linestyle  ="dashed"
p2_noalpha_g0_0001.color="lightblue"
p2_noalpha_g0_0001.label = "ψ₂ - NoAlpha - 0.0001"

p2_noalpha_noreg = RGPsi2Transport(space_interior)
p2_noalpha_noreg.prefix = "results_enatl60_atmp_noalpha_noreg_nowind_obstrack"
p2_noalpha_noreg.linestyle  ="dotted"
p2_noalpha_noreg.color="lightblue"
p2_noalpha_noreg.label = "ψ₂ - NoAlpha - NR"


models = ModelsManagerOBC(
    rg,
    forced_g1000,
    forced_g1,
    forced_noreg,
    p2_g0_01,
    p2_g0_0001,
    p2_noreg,
    p2_noalpha_g0_01,
    p2_noalpha_g0_0001,
    p2_noalpha_noreg,
)
models.save_params = True


gc.collect()
for c in range(n_cycles):
    torch.cuda.reset_peak_memory_stats()
    models.new_cycle()

    psis_ref,psis_filt, times=load_cycle_data_atmp(c)

    u_refs, v_refs = load_cycle_uv_data(c)

    psi0 = psis_filt[0]


    models.reset_time()

    models.setup(psis_filt,times,beta_effect[:,1:-1])

    models.compute_loss(crop(psis_ref[0],b))
    models.compute_uv_losses(crop(u_refs[0],b),crop(v_refs[0],b))
    for n in range(1,n_steps_per_cyle):
        models.step()
        models.compute_loss(crop(psis_ref[n],b))
        models.compute_uv_losses(crop(u_refs[n],b),crop(v_refs[n],b))

    torch.cuda.empty_cache()
    gc.collect()

    max_mem = torch.cuda.max_memory_allocated() / 1024 / 1024
    msg_mem = f"Cycle {step(c + 1, n_cycles)} | Max memory allocated: {max_mem:.1f} MB."
    logger.info(box(msg_mem, style="round"))


In [ ]:
from matplotlib import pyplot as plt

from qgsw import plots

# To Show

# Plots

show_grad = True

fig,axs = plots.subplots(1+show_grad,1,figsize=(15,5+5*show_grad))
plots.set_rowtitles(["RMSE"]+show_grad*[ "Gradient RMSE"] ,axs=axs)
models.plot_loss(loss_name="rmse",ax=axs[0,0])
plots.clamp_ylims(0,1,axs[0,0])
axs[0,0].legend(loc="upper left",prop={'size': 8})
if show_grad:
    models.plot_loss(loss_name="uv_rmse",ax=axs[1,0])
    plots.clamp_ylims(0,1,axs[1,0])
    axs[1,0].legend(loc="upper left",prop={'size': 8})